<a href="https://colab.research.google.com/github/Nefeli-Apostolou/GM-Project---Fraud-Detection/blob/main/Graph_Mining_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---------------------------------------------------------
# 0. Install Dependences
---------------------------------------------------------

In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.0 MB/s eta 0:00:00


In [5]:
import pandas as pd
import networkx as nx
import numpy as np
import torch
import os
from torch_geometric.data import Data

CSV_PATH = "/content/drive/MyDrive/Graph_Mining_Project/eth_tx_last4days_2.csv"



 ---------------------------------------------------------
# 1. Load transaction data
 ---------------------------------------------------------


In [ ]:

tx = pd.read_csv(CSV_PATH)

required_cols = [
    "block_number",
    "hash",
    "from_address",
    "to_address",
    "value",
    "block_timestamp",
]

missing = set(required_cols) - set(tx.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")



 ---------------------------------------------------------
# 2. Clean addresses
 ---------------------------------------------------------


In [ ]:

tx["from_address"] = tx["from_address"].astype(str).str.lower()
tx["to_address"] = tx["to_address"].astype(str).str.lower()

# Remove contract creation transactions
# (transactions with missing destination address)

tx = tx[
    ~tx["to_address"].isin(["nan", "none", "null", ""])
].copy()

# Also remove actual NaN values if present
tx = tx.dropna(subset=["from_address", "to_address"])

print(f"Remaining transactions after removing contract creations: {len(tx):,}")



 ---------------------------------------------------------
# 3. Convert value from Wei to ETH
 ---------------------------------------------------------


In [ ]:

tx["value"] = pd.to_numeric(tx["value"], errors="coerce").fillna(0.0)
tx["value_eth"] = tx["value"] / 1e18



 ---------------------------------------------------------
# 4. Convert timestamp
 ---------------------------------------------------------


In [ ]:

tx["block_timestamp"] = pd.to_datetime(tx["block_timestamp"], errors="coerce")

# fallback if timestamp is Unix seconds
if tx["block_timestamp"].isna().mean() > 0.5:
    tx["block_timestamp"] = pd.to_datetime(
        tx["block_timestamp"],
        unit="s",
        errors="coerce"
    )

tx = tx.dropna(subset=["block_timestamp"])

tx["timestamp_unix"] = tx["block_timestamp"].astype("int64") // 10**9



 ---------------------------------------------------------
# 5. Create node mapping
 ---------------------------------------------------------


In [ ]:

all_addresses = pd.concat(
    [tx["from_address"], tx["to_address"]],
    ignore_index=True
).drop_duplicates()

node_id = pd.Series(
    data=np.arange(len(all_addresses), dtype=np.int64),
    index=all_addresses.values
)

tx["src"] = tx["from_address"].map(node_id).astype(np.int64)
tx["dst"] = tx["to_address"].map(node_id).astype(np.int64)

num_nodes = len(node_id)

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of directed edges: {len(tx):,}")



 ---------------------------------------------------------
# 6. Build edge_index
 ---------------------------------------------------------


In [ ]:

edge_index = torch.tensor(
    tx[["src", "dst"]].values.T,
    dtype=torch.long
)

# Edge attributes: value_eth and timestamp only
edge_attr_np = tx[["value_eth", "timestamp_unix"]].copy()
edge_attr_np = edge_attr_np.replace([np.inf, -np.inf], np.nan).fillna(0.0)

edge_attr = torch.tensor(
    edge_attr_np.values,
    dtype=torch.float
)



 ---------------------------------------------------------
# 7. Basic flow features
 ---------------------------------------------------------


In [ ]:

out_stats = tx.groupby("src").agg(
    mean_send_amount=("value_eth", "mean"),
    max_send_amount=("value_eth", "max"),
)

in_stats = tx.groupby("dst").agg(
    mean_recv_amount=("value_eth", "mean"),
    max_recv_amount=("value_eth", "max"),
)

# These are used internally to compute ratios, but dropped later
tmp_out = tx.groupby("src").agg(
    send_num=("hash", "count"),
    send_amount=("value_eth", "sum"),
)

tmp_in = tx.groupby("dst").agg(
    recv_num=("hash", "count"),
    recv_amount=("value_eth", "sum"),
)

node_features = pd.DataFrame(index=np.arange(num_nodes))

node_features = node_features.join(out_stats, how="left")
node_features = node_features.join(in_stats, how="left")
node_features = node_features.join(tmp_out, how="left")
node_features = node_features.join(tmp_in, how="left")
node_features = node_features.fillna(0.0)

node_features["total_tx"] = (
    node_features["send_num"] + node_features["recv_num"]
)

node_features["total_amount"] = (
    node_features["send_amount"] + node_features["recv_amount"]
)

node_features["out_ratio"] = (
    node_features["send_num"] /
    node_features["total_tx"].replace(0, np.nan)
).fillna(0.0)

node_features["in_ratio"] = (
    node_features["recv_num"] /
    node_features["total_tx"].replace(0, np.nan)
).fillna(0.0)

node_features["amount_out_ratio"] = (
    node_features["send_amount"] /
    node_features["total_amount"].replace(0, np.nan)
).fillna(0.0)

node_features["amount_in_ratio"] = (
    node_features["recv_amount"] /
    node_features["total_amount"].replace(0, np.nan)
).fillna(0.0)



 ---------------------------------------------------------
# 8. Structural graph features
 ---------------------------------------------------------


In [ ]:

# We aggregate repeated transactions between the same pair of addresses.
# This makes centrality computation much cheaper.

edge_weights = (
    tx.groupby(["src", "dst"])
    .agg(
        tx_count=("hash", "count"),
        total_value=("value_eth", "sum")
    )
    .reset_index()
)

print(f"Aggregated directed edges: {len(edge_weights):,}")

# Directed graph for PageRank and directed degrees
G_dir = nx.DiGraph()
G_dir.add_nodes_from(range(num_nodes))
G_dir.add_weighted_edges_from(
    edge_weights[["src", "dst", "tx_count"]].itertuples(index=False, name=None),
    weight="weight"
)
'''
# Undirected graph for clustering, eigenvector, betweenness
G_undir = nx.Graph()
G_undir.add_nodes_from(range(num_nodes))
G_undir.add_weighted_edges_from(
    edge_weights[["src", "dst", "tx_count"]].itertuples(index=False, name=None),
    weight="weight"
)
'''
# Directed unique-neighbor degrees
in_degree_dict = dict(G_dir.in_degree())
out_degree_dict = dict(G_dir.out_degree())

node_features["in_degree"] = pd.Series(in_degree_dict)
node_features["out_degree"] = pd.Series(out_degree_dict)

# PageRank
print("Computing PageRank...")
pagerank_dict = nx.pagerank(
    G_dir,
    alpha=0.85,
    max_iter=100,
    tol=1e-06,
    weight="weight"
)

node_features["pagerank"] = pd.Series(pagerank_dict)

'''
# Clustering coefficient
print("Computing clustering coefficient...")
clustering_dict = nx.clustering(
    G_undir,
    weight="weight"
)

node_features["clustering_coefficient"] = pd.Series(clustering_dict)


# Eigenvector centrality
# This can be slow on very large graphs.
print("Computing eigenvector centrality...")
try:
    eigen_dict = nx.eigenvector_centrality(
        G_undir,
        max_iter=300,
        tol=1e-06,
        weight="weight"
    )
except nx.PowerIterationFailedConvergence:
    print("Eigenvector centrality did not converge; filling with zeros.")
    eigen_dict = {i: 0.0 for i in range(num_nodes)}

node_features["eigenvector_centrality"] = pd.Series(eigen_dict)
'''

# Approximate betweenness centrality
# Exact betweenness is usually impossible on million-node graphs.
# Increase k for better accuracy, decrease k for speed.
print("Computing approximate betweenness centrality...")
BETWEENNESS_SAMPLE_SIZE = min(1000, num_nodes)

betweenness_dict = nx.betweenness_centrality(
    G_undir,
    k=BETWEENNESS_SAMPLE_SIZE,
    normalized=True,
    weight=None,
    seed=42
)

node_features["betweenness"] = pd.Series(betweenness_dict)

node_features = node_features.fillna(0.0)



 ---------------------------------------------------------
# 9. Temporal node features
 ---------------------------------------------------------


In [ ]:

events_out = tx[["src", "timestamp_unix", "block_timestamp"]].copy()
events_out = events_out.rename(columns={"src": "node"})

events_in = tx[["dst", "timestamp_unix", "block_timestamp"]].copy()
events_in = events_in.rename(columns={"dst": "node"})

events = pd.concat([events_out, events_in], ignore_index=True)
events = events.sort_values(["node", "timestamp_unix"])

events["node_inter_tx_time"] = (
    events.groupby("node")["timestamp_unix"].diff()
)

events["node_inter_tx_time"] = (
    events["node_inter_tx_time"]
    .replace([np.inf, -np.inf], np.nan)
)

inter_stats = events.groupby("node")["node_inter_tx_time"].agg(
    mean_inter_tx_time="mean",
    std_inter_tx_time="std"
).fillna(0.0)

node_features = node_features.join(inter_stats, how="left")
node_features = node_features.fillna(0.0)

mu = node_features["mean_inter_tx_time"]
sigma = node_features["std_inter_tx_time"]

node_features["burstiness"] = (
    (sigma - mu) /
    (sigma + mu).replace(0, np.nan)
).fillna(0.0)

time_span_stats = events.groupby("node")["timestamp_unix"].agg(
    first_tx_time="min",
    last_tx_time="max",
    tx_event_count="count"
)

time_span_stats["active_seconds"] = (
    time_span_stats["last_tx_time"] - time_span_stats["first_tx_time"]
)

time_span_stats["active_days"] = (
    time_span_stats["active_seconds"] / 86400
)

active_hours = (time_span_stats["active_seconds"] / 3600).clip(lower=1)

time_span_stats["tx_per_hour"] = (
    time_span_stats["tx_event_count"] / active_hours
)

node_features = node_features.join(
    time_span_stats[
        [
            "active_days",
            "tx_per_hour",
        ]
    ],
    how="left"
)

node_features = node_features.fillna(0.0)

# Night activity: 00:00–06:00 UTC
events["hour"] = events["block_timestamp"].dt.hour
events["is_night"] = events["hour"].between(0, 5).astype(float)

night_stats = events.groupby("node")["is_night"].mean()
night_stats.name = "night_activity_ratio"

node_features = node_features.join(night_stats, how="left")
node_features["night_activity_ratio"] = (
    node_features["night_activity_ratio"].fillna(0.0)
)



 ---------------------------------------------------------
# 10. Remove unwanted/redundant features
 ---------------------------------------------------------


In [ ]:

drop_cols = [
    "send_num",
    "recv_num",
    "send_amount",
    "recv_amount",
]

node_features = node_features.drop(columns=drop_cols, errors="ignore")



 ---------------------------------------------------------
# 11. Clean and log-transform selected features
 ---------------------------------------------------------


In [ ]:

node_features = node_features.replace([np.inf, -np.inf], 0.0)
node_features = node_features.fillna(0.0)

# Do not log-transform ratios, burstiness, PageRank, clustering, eigenvector, betweenness.
log_cols = [
    "mean_send_amount",
    "mean_recv_amount",
    "max_send_amount",
    "max_recv_amount",
    "total_tx",
    "total_amount",
    "in_degree",
    "out_degree",
    "mean_inter_tx_time",
    "std_inter_tx_time",
    "active_days",
    "tx_per_hour",
]

for col in log_cols:
    if col in node_features.columns:
        node_features[col] = np.log1p(node_features[col].clip(lower=0))

x = torch.tensor(
    node_features.values,
    dtype=torch.float
)

print("Node feature matrix shape:", x.shape)
print("Node features:")
print(node_features.columns.tolist())

 ---------------------------------------------------------
# 12. Create PyTorch Geometric graph object
 ---------------------------------------------------------


In [ ]:

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    num_nodes=num_nodes
)

print(data)

print("\nNode feature matrix shape:")
print(data.x.shape)

print("\nEdge index shape:")
print(data.edge_index.shape)

print("\nEdge attribute shape:")
print(data.edge_attr.shape)



---------------------------------------------------------
# 13. Save outputs
---------------------------------------------------------


In [ ]:

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Main PyTorch Geometric graph
torch.save(
    data,
    os.path.join(OUTPUT_DIR, "ethereum_full_graph.pt")
)

# Address ↔ node ID mapping
node_mapping_df = pd.DataFrame({
    "address": node_id.index,
    "node_id": node_id.values
})

node_mapping_df.to_csv(
    os.path.join(OUTPUT_DIR, "node_mapping.csv"),
    index=False
)

# Human-readable node features
node_features_export = node_features.copy()
node_features_export["node_id"] = node_features_export.index

node_features_export.to_csv(
    os.path.join(OUTPUT_DIR, "node_features.csv"),
    index=False
)

# Aggregated edge list
edge_weights.to_csv(
    os.path.join(OUTPUT_DIR, "aggregated_edges.csv"),
    index=False
)

print("\nSaved files:")
print("- ethereum_full_graph.pt")
print("- node_mapping.csv")
print("- node_features.csv")
print("- aggregated_edges.csv")

# Aggregated edge list:
# Each row represents a unique directed connection between two addresses.
#
# Columns:
# - src: source node ID
# - dst: destination node ID
# - tx_count: number of transactions between the two nodes
# - total_value: total ETH transferred across those transactions
#
# This compressed graph is useful for:
# - temporal motif extraction
# - community detection
# - graph visualization
# - debugging and inspection
# - temporal aggregation
# - classical network analysis
#
# Using aggregated edges is much more memory-efficient than repeatedly
# processing the raw transaction table.

In [10]:
# ---------------------------------------------------------
# 1. Load saved Ethereum graph files
# ---------------------------------------------------------

def load_ethereum_graph_files(
    output_dir="/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn"
):
    graph_path = os.path.join(output_dir, "ethereum_full_graph.pt")
    mapping_path = os.path.join(output_dir, "node_mapping.csv")
    features_path = os.path.join(output_dir, "node_features.csv")
    edges_path = os.path.join(output_dir, "aggregated_edges.csv")

    data = torch.load(
        graph_path,
        map_location="cpu",
        weights_only=False
    )

    node_mapping = pd.read_csv(mapping_path)
    node_features = pd.read_csv(features_path)
    aggregated_edges = pd.read_csv(edges_path)

    node_mapping["address"] = (
        node_mapping["address"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    return data, node_mapping, node_features, aggregated_edges

data, node_mapping, node_features, aggregated_edges = load_ethereum_graph_files()

print(data)
print("Node mapping shape:", node_mapping.shape)
print("Node features shape:", node_features.shape)
print("Aggregated edges shape:", aggregated_edges.shape)

Data(x=[1746059, 20], edge_index=[2, 5986025], edge_attr=[5986025, 2], num_nodes=1746059)
Node mapping shape: (1746059, 2)
Node features shape: (1746059, 21)
Aggregated edges shape: (2842727, 4)


In [12]:
# ---------------------------------------------------------
# 2. Check overlap with online scam-wallet CSV
# ---------------------------------------------------------

SCAM_CSV_PATH = "/content/drive/MyDrive/DD_Project/checkcrypto_eth_scam_wallets.csv"

scam_df = pd.read_csv(SCAM_CSV_PATH)

required_cols = ["id", "address", "reportCount", "lastReported", "network"]
missing = set(required_cols) - set(scam_df.columns)

if missing:
    raise ValueError(f"Missing columns in scam CSV: {missing}")

# Normalize scam addresses
scam_df["address"] = (
    scam_df["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Optional: keep only Ethereum entries if the network column contains multiple chains
scam_eth = scam_df[
    scam_df["network"].astype(str).str.lower().str.contains("eth")
].copy()

print("Total scam accounts in CSV:", len(scam_df))
print("Ethereum scam accounts in CSV:", len(scam_eth))

# ---------------------------------------------------------
# 3. Match scam addresses against graph nodes
# ---------------------------------------------------------

graph_addresses = set(node_mapping["address"])
scam_addresses = set(scam_eth["address"])

overlap_addresses = graph_addresses.intersection(scam_addresses)

print("Scam accounts present in graph:", len(overlap_addresses))
print(
    "Percentage of Ethereum scam list found in graph:",
    len(overlap_addresses) / len(scam_addresses) * 100
)

# ---------------------------------------------------------
# 4. Create overlap table with node IDs
# ---------------------------------------------------------

overlap_df = node_mapping[
    node_mapping["address"].isin(overlap_addresses)
].merge(
    scam_eth,
    on="address",
    how="left"
)

print("Overlap table shape:", overlap_df.shape)
display(overlap_df.head())

# ---------------------------------------------------------
# 5. Save matched scam accounts
# ---------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn"

overlap_df.to_csv(
    os.path.join(OUTPUT_DIR, "known_scam_accounts_in_graph.csv"),
    index=False
)

print("Saved:")
print("- known_scam_accounts_in_graph.csv")

Total scam accounts in CSV: 5463
Ethereum scam accounts in CSV: 5463
Scam accounts present in graph: 115
Percentage of Ethereum scam list found in graph: 2.857142857142857
Overlap table shape: (173, 6)


,address,node_id,id,reportCount,lastReported,network
0,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,37,cmgb64u3rn8ln3oi2t4d8js7n,1,2025-10-03T18:19:18.759Z,eth
1,0xeba88149813bec1cccccfdb0dacefaaa5de94cb1,78,cm4kso2drman9gbg46lmejsg1,2,2025-06-16T06:43:42.566Z,eth
2,0xeba88149813bec1cccccfdb0dacefaaa5de94cb1,78,cm4kso2drman9gbg46lmejsg1,2,2025-06-16T06:43:42.566Z,eth
3,0xeba88149813bec1cccccfdb0dacefaaa5de94cb1,78,cm2xb5qazaoc24doo8lpvplix,1,2024-10-31T12:55:01.740Z,eth
4,0xf70da97812cb96acdf810712aa562db8dfa3dbef,170,cm1qqj1sw6cta5u0sg36dcyn6,1,2024-10-01T17:51:11.840Z,eth


Saved:
- known_scam_accounts_in_graph.csv
